<a href="https://colab.research.google.com/github/KalaiselvamK23/MS-Elevate-Microsoft-Azure/blob/main/Sleep_efficiency%2Cloaded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter


file_path = "Sleep_Efficiency.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "equilibriumm/sleep-efficiency",
  file_path,

  # documenation for more information:
  # https://github.com/Kaggle/kaggle/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

/tmp/ipython-input-1201118534.py:11: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'sleep-efficiency' dataset.
First 5 records:    ID  Age  Gender              Bedtime          Wakeup time  Sleep duration  \
0   1   65  Female  2021-03-06 01:00:00  2021-03-06 07:00:00             6.0   
1   2   69    Male  2021-12-05 02:00:00  2021-12-05 09:00:00             7.0   
2   3   40  Female  2021-05-25 21:30:00  2021-05-25 05:30:00             8.0   
3   4   40  Female  2021-11-03 02:30:00  2021-11-03 08:30:00             6.0   
4   5   57    Male  2021-03-13 01:00:00  2021-03-13 09:00:00             8.0   

   Sleep efficiency  REM sleep percentage  Deep sleep percentage  \
0              0.88                    18                     70   
1              0.66                    19                     28   
2              0.89                    20                     70   
3              0.51                    23                     25   
4              0.76                    27                     55   

   Light sleep percent

In [ ]:

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


data = df.copy()

# Encode Categorical
le = LabelEncoder()
data['Smoking status_encoded'] = le.fit_transform(data['Smoking status'])
data['Gender_encoded'] = le.fit_transform(data['Gender'])


data.drop(['Smoking status', 'Gender', 'Bedtime', 'Wakeup time'], axis=1, inplace=True)

# 2. Split Features and Target
X = data[['Age','Gender_encoded','REM sleep percentage','Deep sleep percentage','Light sleep percentage','Awakenings','Smoking status_encoded','Caffeine consumption','Alcohol consumption','Exercise frequency']]
y = data['Sleep efficiency'] # 1D Series is better for Sklearn

# 3. Impute & Scale
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

train_x, test_x, train_y, test_y = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
train_x_scaled = scaler.fit_transform(train_x)
test_x_scaled = scaler.transform(test_x)

# 4. Train Best Model (Random Forest usually performs best here)
final_model = RandomForestRegressor(n_estimators=100, random_state=42)
final_model.fit(train_x_scaled, train_y)

print("Model Training Complete.")

Model Training Complete.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create input fields
age_in = widgets.IntSlider(value=30, min=10, max=80, description='Age:')
gender_in = widgets.Dropdown(options=[('Male', 1), ('Female', 0)], description='Gender:')
rem_in = widgets.FloatSlider(value=20, min=0, max=100, description='REM %:')
deep_in = widgets.FloatSlider(value=50, min=0, max=100, description='Deep %:')
light_in = widgets.FloatSlider(value=30, min=0, max=100, description='Light %:')
awake_in = widgets.IntSlider(value=1, min=0, max=10, description='Awakenings:')
smoke_in = widgets.Dropdown(options=[('Yes', 1), ('No', 0)], description='Smoker:')
caff_in = widgets.FloatSlider(value=0, min=0, max=500, description='Caffeine (mg):')
alc_in = widgets.FloatSlider(value=0, min=0, max=10, description='Alcohol (oz):')
exe_in = widgets.IntSlider(value=3, min=0, max=7, description='Exercise/wk:')

button = widgets.Button(description="Predict Sleep Efficiency", button_style='success')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        # Arrange inputs for the model
        user_data = np.array([[age_in.value, gender_in.value, rem_in.value, deep_in.value,
                               light_in.value, awake_in.value, smoke_in.value,
                               caff_in.value, alc_in.value, exe_in.value]])


        user_scaled = scaler.transform(user_data)

        # Predict
        prediction = final_model.predict(user_scaled)
        print(f"--- PREDICTION ---")
        print(f"Estimated Sleep Efficiency: {prediction[0]:.2f}")


        if prediction[0] > 0.85:
            print("Status: Excellent Sleep Quality! 🌙")
        elif prediction[0] > 0.70:
            print("Status: Good Sleep Quality. 👍")
        else:
            print("Status: Poor Sleep Quality. Consider adjusting habits. ⚠️")

button.on_click(on_button_clicked)

# Layout
ui = widgets.VBox([
    widgets.HBox([age_in, gender_in]),
    widgets.HBox([rem_in, deep_in, light_in]),
    widgets.HBox([awake_in, smoke_in]),
    widgets.HBox([caff_in, alc_in, exe_in]),
    button, output
])

display(ui)

In [1]:
import joblib
import pandas as pd

model = joblib.load("../models/sleep_efficiency_model.pkl")
scaler = joblib.load("../models/scaler.pkl")

features = [
    "Age",
    "Gender",
    "Sleep duration",
    "REM sleep percentage",
    "Deep sleep percentage",
    "Light sleep percentage",
    "Awakenings",
    "Caffeine consumption",
    "Alcohol consumption",
    "Smoking status",
    "Exercise frequency",
    "Bedtime_hour",
    "Wakeup_hour"
]

input_1 = pd.DataFrame([[
    30, 1, 8, 25, 25, 50, 0, 0, 0, 0, 5, 22, 6
]], columns=features)

input_2 = pd.DataFrame([[
    30, 1, 8, 25, 80, 10, 0, 0, 0, 0, 5, 22, 6
]], columns=features)

scaled_1 = scaler.transform(input_1)
scaled_2 = scaler.transform(input_2)

prediction_1 = model.predict(scaled_1)[0]
prediction_2 = model.predict(scaled_2)[0]

print("Prediction 1:", prediction_1)
print("Prediction 2:", prediction_2)
print("Difference:", prediction_2 - prediction_1)

Prediction 1: 0.7298392383814426
Prediction 2: 0.7298392383814426
Difference: 0.0


c:\Users\kalai\OneDrive\Documents\Projects\Sleep-Quality-Tracker\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\kalai\OneDrive\Documents\Projects\Sleep-Quality-Tracker\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(


In [2]:
print("Model type:", type(model))
print("Number of features:", model.n_features_in_)
print("Feature names:")
print(model.feature_names_in_)

print("\nFeature importances:")
for name, importance in zip(
    model.feature_names_in_,
    model.feature_importances_
):
    print(f"{name}: {importance:.6f}")

Model type: <class 'sklearn.ensemble._gb.GradientBoostingRegressor'>
Number of features: 13
Feature names:
['Age' 'Gender' 'Sleep duration' 'REM sleep percentage'
 'Deep sleep percentage' 'Light sleep percentage' 'Awakenings'
 'Caffeine consumption' 'Alcohol consumption' 'Smoking status'
 'Exercise frequency' 'Bedtime_hour' 'Wakeup_hour']

Feature importances:
Age: 0.013339
Gender: 0.000000
Sleep duration: 0.000000
REM sleep percentage: 0.001077
Deep sleep percentage: 0.358165
Light sleep percentage: 0.436998
Awakenings: 0.150892
Caffeine consumption: 0.000000
Alcohol consumption: 0.005507
Smoking status: 0.028237
Exercise frequency: 0.003952
Bedtime_hour: 0.000109
Wakeup_hour: 0.001726


In [3]:
print("\nModel parameters:")
print(model.get_params())


Model parameters:
{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'deprecated', 'init': None, 'learning_rate': 0.03, 'loss': 'squared_error', 'max_depth': 2, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 150, 'n_iter_no_change': None, 'random_state': 42, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}


In [4]:
print("Scaled input 1:")
print(scaled_1)

print("\nScaled input 2:")
print(scaled_2)

print("\nDifference between scaled inputs:")
print(scaled_2 - scaled_1)

Scaled input 1:
[[-0.78555483  1.02524942  0.64056173  0.68076057 -1.77851028  1.65412491
  -1.21317767 -0.82385835 -0.71374035 -0.67936622  2.29770688  1.03898894
  -0.52696199]]

Scaled input 2:
[[-0.78555483  1.02524942  0.64056173  0.68076057  1.73830295 -0.94892258
  -1.21317767 -0.82385835 -0.71374035 -0.67936622  2.29770688  1.03898894
  -0.52696199]]

Difference between scaled inputs:
[[ 0.          0.          0.          0.          3.51681323 -2.60304749
   0.          0.          0.          0.          0.          0.
   0.        ]]


In [5]:
print("Scaler feature names:")
print(scaler.feature_names_in_)

Scaler feature names:
['Age' 'Gender' 'Sleep duration' 'REM sleep percentage'
 'Deep sleep percentage' 'Light sleep percentage' 'Awakenings'
 'Caffeine consumption' 'Alcohol consumption' 'Smoking status'
 'Exercise frequency' 'Bedtime_hour' 'Wakeup_hour']


In [6]:
import numpy as np

predictions_1 = np.array([
    tree[0].predict(scaled_1)[0]
    for tree in model.estimators_
])

predictions_2 = np.array([
    tree[0].predict(scaled_2)[0]
    for tree in model.estimators_
])

print("First 10 tree predictions - Input 1:")
print(predictions_1[:10])

print("\nFirst 10 tree predictions - Input 2:")
print(predictions_2[:10])

print("\nNumber of trees with different predictions:")
print(np.sum(predictions_1 != predictions_2))

print("\nTotal trees:")
print(len(model.estimators_))

First 10 tree predictions - Input 1:
[-0.14900345 -0.14453334  0.0961751   0.09328984  0.09049115  0.08777641
  0.08514312  0.08258883 -0.11678046  0.07770783]

First 10 tree predictions - Input 2:
[-0.14900345 -0.14453334  0.0961751   0.09328984  0.09049115  0.08777641
  0.08514312  0.08258883 -0.11678046  0.07770783]

Number of trees with different predictions:
0

Total trees:
150


In [7]:
leaves_1 = model.apply(scaled_1)
leaves_2 = model.apply(scaled_2)

print("Different tree paths:")
print(np.sum(leaves_1 != leaves_2))

print("\nTotal trees:")
print(leaves_1.shape[1])

print("\nFirst 20 leaf indices - Input 1:")
print(leaves_1[0, :20])

print("\nFirst 20 leaf indices - Input 2:")
print(leaves_2[0, :20])

Different tree paths:
0

Total trees:
150

First 20 leaf indices - Input 1:
[2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]

First 20 leaf indices - Input 2:
[2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]


In [8]:
print("Model training feature ranges:")

for feature, min_val, max_val in zip(
    model.feature_names_in_,
    scaler.mean_ - 3 * scaler.scale_,
    scaler.mean_ + 3 * scaler.scale_
):
    print(f"{feature}: {min_val:.2f} to {max_val:.2f}")

Model training feature ranges:
Age: 1.01 to 79.56
Gender: -1.01 to 1.99
Sleep duration: 4.82 to 10.06
REM sleep percentage: 12.04 to 33.16
Deep sleep percentage: 5.90 to 99.73
Light sleep percentage: -21.52 to 70.68
Awakenings: -2.39 to 5.65
Caffeine consumption: -65.67 to 115.39
Alcohol consumption: -3.63 to 5.90
Smoking status: -1.08 to 1.71
Exercise frequency: -2.56 to 6.00
Bedtime_hour: -20.70 to 42.73
Wakeup_hour: 1.21 to 12.83


In [9]:
tree = model.estimators_[0, 0]

print("Tree depth:", tree.get_depth())
print("Number of leaves:", tree.get_n_leaves())

print("\nFeatures used by first tree:")

for feature_index in tree.tree_.feature:
    if feature_index >= 0:
        print(model.feature_names_in_[feature_index])

Tree depth: 2
Number of leaves: 4

Features used by first tree:
Deep sleep percentage
Smoking status
Awakenings


In [10]:
print("\nFirst tree split details:")

for i in range(tree.tree_.node_count):
    feature_index = tree.tree_.feature[i]

    if feature_index >= 0:
        feature_name = model.feature_names_in_[feature_index]
        threshold = tree.tree_.threshold[i]

        print(
            f"Node {i}: "
            f"{feature_name} <= {threshold:.4f}"
        )


First tree split details:
Node 0: Deep sleep percentage <= 44.0000
Node 1: Smoking status <= 0.5000
Node 4: Awakenings <= 1.5000


In [11]:
raw_prediction_1 = model.predict(input_1)[0]
raw_prediction_2 = model.predict(input_2)[0]

print("Raw prediction 1:", raw_prediction_1)
print("Raw prediction 2:", raw_prediction_2)
print("Difference:", raw_prediction_2 - raw_prediction_1)

Raw prediction 1: 0.6623193133596262
Raw prediction 2: 0.9192423608230175
Difference: 0.25692304746339123


In [1]:
df_check = pd.read_csv("data/sleep_efficiency_cleaned.csv")

X_check = df_check.drop(columns=["Sleep efficiency"])

predictions = model.predict(X_check)

df_check["Predicted Efficiency"] = predictions

df_check.sort_values(
    "Predicted Efficiency",
    ascending=False
).head(10)


NameError: name 'pd' is not defined